# Picotron All-in-One Training, Alignment, & PEFT Pipeline
This notebook verifies the complete suite of Picotron features in a single notebook session:
1. **Environment Setup**: Clones the repository and installs package dependencies.
2. **Configuration**: Writes custom configurations for SFT, LoRA/DoRA, DPO, GRPO, and PPO.
3. **Mock Data Generation**: Creates a mock dataset for preference and alignment tasks.
4. **Execution Pipeline**: Runs pre-training verification, adapter fine-tuning, preference alignment, and reinforcement learning.

## Step 1: Environment Setup

In [ ]:
# Set base directory and clone fresh repository
%cd /kaggle/working
!rm -rf picotron
!pip uninstall -y picotron
!git clone https://github.com/Syntropy-AI-Labs/picotron.git

# Install in editable mode
%cd picotron
!pip install -e .

## Step 2: Write Training Configuration

In [ ]:
%%writefile config_all_in_one.yaml
model:
  vocab_size: 100
  hidden_size: 16
  num_hidden_layers: 2
  num_attention_heads: 2
  num_key_value_heads: 1
  intermediate_size: 32
  max_position_embeddings: 32
  norm_type: "rms"
  activation_type: "silu"
  rms_norm_eps: 0.000005
  bias: false

parallel:
  dp_size: 1
  zero_stage: 0

data:
  dataset_path: "data/all_in_one_train.bin"
  validation_dataset_path: "data/all_in_one_val.bin"
  sequence_length: 32
  micro_batch_size: 2
  num_workers: 1

train:
  learning_rate: 0.001
  min_learning_rate: 0.0001
  weight_decay: 0.01
  max_steps: 10
  warmup_steps: 2
  grad_accum_steps: 1
  seed: 42
  compile: false
  use_cuda_graphs: false
  mixed_precision: "bf16"
  save_checkpoint: false
  checkpoint_dir: "checkpoints/all_in_one"

## Step 3: Run Full Pipeline Execution
Loads a single model architecture and executes:
1. **Supervised Fine-Tuning (SFT)** with response masking.
2. **PEFT adapter injection (LoRA / DoRA)** and adapter weight merging.
3. **Direct Preference Optimization (DPO)** using preference dataset pairs.
4. **Group Relative Policy Optimization (GRPO)** relative alignment.
5. **Proximal Policy Optimization (PPO)** Reinforcement Learning loop.

In [ ]:
# Force clear stale imports cache from Jupyter kernel memory
import sys
for mod in list(sys.modules.keys()):
    if "picotron" in mod:
        del sys.modules[mod]

import torch
from torch.utils.data import TensorDataset, DataLoader

from picotron.config import load_config_from_yaml
from picotron.models.llama import LLaMAModel
from picotron.peft.peft_model import PeftModel
from picotron.peft.sft_trainer import SFTTrainer
from picotron.rlhf.trainer import PreferenceTrainer
from picotron.rlhf.dataset import PreferenceDataset
from picotron.rlhf.ppo import ActorCritic, RewardModel, PPORolloutBuffer, PPOTrainer

cfg = load_config_from_yaml("config_all_in_one.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing training stages on: {device.upper()}")

# -----------------------------------------------------------------
# 1. MODEL INITIALIZATION
# -----------------------------------------------------------------
print("\n--- Stage 1: Model Initialization ---")
base_model = LLaMAModel(cfg.model).to(device)
print("Model built successfully.")

# -----------------------------------------------------------------
# 2. ADAPTER INJECTION (LoRA/DoRA)
# -----------------------------------------------------------------
print("\n--- Stage 2: Adapter Injection (DoRA) ---")
peft_model = PeftModel(base_model, target_modules=["q_proj", "v_proj"], r=4, use_dora=True)
print(f"Injected target adapters. State dict parameters: {list(peft_model.get_adapter_state_dict().keys())}")

# -----------------------------------------------------------------
# 3. SUPERVISED FINE-TUNING (SFT)
# -----------------------------------------------------------------
print("\n--- Stage 3: Supervised Fine-Tuning (SFT) ---")
x_sft = torch.randint(0, cfg.model.vocab_size, (4, 32), device=device)
y_sft = torch.randint(0, cfg.model.vocab_size, (4, 32), device=device)
# Mask out prompt tokens by setting label elements to -100
y_sft[:, :16] = -100
sft_dataset = TensorDataset(x_sft, y_sft)
sft_dataloader = DataLoader(sft_dataset, batch_size=2)

sft_trainer = SFTTrainer(config=cfg, model=peft_model, train_dataloader=sft_dataloader)
sft_loss = sft_trainer.train_step(x_sft, y_sft)
print(f"SFT single-step response-masked loss: {sft_loss:.4f}")

# Merge adapters back to base layer weights
peft_model.merge_adapters()
print("Merged DoRA adapter weights into base layer.")
peft_model.unmerge_adapters()
print("Unmerged DoRA adapter weights.")

# -----------------------------------------------------------------
# 4. PREFERENCE ALIGNMENT (DPO/GRPO)
# -----------------------------------------------------------------
print("\n--- Stage 4: Preference Alignment (DPO/GRPO) ---")
mock_pref_data = [
    {
        "prompt": [1, 2, 3, 4],
        "chosen": [5, 6, 7, 8],
        "rejected": [9, 10, 11, 12]
    }
] * 10
pref_dataset = PreferenceDataset(mock_pref_data, max_length=16)
pref_dataloader = DataLoader(pref_dataset, batch_size=2)

# Reference Model for Logits Baseline
ref_model = LLaMAModel(cfg.model).to(device)

# A. Run Direct Preference Optimization (DPO)
print("Running DPO Trainer...")
dpo_trainer = PreferenceTrainer(
    config=cfg,
    model=base_model,
    ref_model=ref_model,
    train_dataloader=pref_dataloader,
    mode="dpo"
)
batch = next(iter(pref_dataloader))
dpo_loss = dpo_trainer.train_step(batch)
print(f"DPO step preference loss: {dpo_loss:.4f}")

# B. Run Group Relative Policy Optimization (GRPO)
print("Running GRPO Trainer...")
grpo_trainer = PreferenceTrainer(
    config=cfg,
    model=base_model,
    ref_model=ref_model,
    train_dataloader=pref_dataloader,
    mode="grpo"
)
grpo_loss = grpo_trainer.train_step(batch)
print(f"GRPO step policy loss: {grpo_loss:.4f}")

# -----------------------------------------------------------------
# 5. REINFORCEMENT LEARNING WITH PPO
# -----------------------------------------------------------------
print("\n--- Stage 5: Reinforcement Learning (PPO) ---")
policy_actor_critic = ActorCritic(base_model).to(device)
reward_estimator = RewardModel(ref_model).to(device)
ppo_optimizer = torch.optim.AdamW(policy_actor_critic.parameters(), lr=0.0001)

ppo_trainer = PPOTrainer(
    actor_critic=policy_actor_critic,
    ref_model=ref_model,
    optimizer=ppo_optimizer
)

# Insert mock experiences into Rollout Buffer
buffer = PPORolloutBuffer()
q = torch.randint(0, cfg.model.vocab_size, (2, 8), device=device)
r = torch.randint(0, cfg.model.vocab_size, (2, 8), device=device)
logp = torch.randn((2, 8), device=device)
val = torch.randn((2, 8), device=device)
rew = torch.randn((2,), device=device)
mask = torch.ones((2, 8), device=device)
buffer.insert(q, r, logp, val, rew, mask)

ppo_loss = ppo_trainer.train_step_ppo(buffer.get_batches())
print(f"PPO trainer update step loss: {ppo_loss:.4f}")

print("\n--- ALL PIPELINE STAGES COMPLETED SUCCESSFULLY ---")